In [3]:
import pandas as pd
from pathlib import Path

# ── Paths (hardcoded) ─────────────────────────────────────────────────────────
RAW = Path("/data/raw")
OUT = Path("/data/interim")
OUT.mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────────
TIER1 = {
    "BLZ","CRI","CUB","DOM","SLV","GTM","GUY",
    "HTI","HND","JAM","NIC","PAN","PRI","SUR","TTO","BHS","BRB"
}
PRESSURE = {"COL", "VEN"}
ALL_COUNTRIES = TIER1 | PRESSURE
YEARS = list(range(2019, 2025))

# FTS location names → ISO3
NAME_TO_ISO3 = {
    "Bahamas":                                "BHS",
    "Barbados":                               "BRB",
    "Belize":                                 "BLZ",
    "Colombia":                               "COL",
    "Costa Rica":                             "CRI",
    "Cuba":                                   "CUB",
    "Dominican Republic":                     "DOM",
    "El Salvador":                            "SLV",
    "Guatemala":                              "GTM",
    "Guyana":                                 "GUY",
    "Haiti":                                  "HTI",
    "Honduras":                               "HND",
    "Jamaica":                                "JAM",
    "Nicaragua":                              "NIC",
    "Panama":                                 "PAN",
    "Puerto Rico":                            "PRI",
    "Suriname":                               "SUR",
    "Trinidad and Tobago":                    "TTO",
    "Venezuela (Bolivarian Republic of)":     "VEN",
    "Venezuela":                              "VEN",
}


def load_year(year):
    path = RAW / f"FTS_Global_summary_countries_{year}.xlsx"
    df = pd.read_excel(path, sheet_name="Export data", header=None)

    # Real headers are in row 1 (row 0 is blank)
    df.columns = df.iloc[1]
    df = df.iloc[2:].reset_index(drop=True)
    df.columns = ["location", "funding_usd", "pledges_usd"]

    df = df[df["location"].notna()].copy()
    df["iso3"] = df["location"].map(NAME_TO_ISO3)
    df["funding_usd"] = pd.to_numeric(df["funding_usd"], errors="coerce").fillna(0)
    df["year"] = year

    return df[df["iso3"].isin(ALL_COUNTRIES)][["iso3", "year", "funding_usd"]]


def main():
    print("Loading OCHA FTS files...")
    frames = []
    for year in YEARS:
        df = load_year(year)
        print(f"  {year}: {len(df)} countries found")
        frames.append(df)

    raw = pd.concat(frames, ignore_index=True)

    # ── Build full grid and fill missing with 0 ───────────────────────────────
    full_index = pd.MultiIndex.from_product(
        [sorted(ALL_COUNTRIES), YEARS], names=["iso3", "year"]
    )
    annual = (
        raw.groupby(["iso3", "year"])["funding_usd"]
        .sum()
        .reindex(full_index, fill_value=0)
        .reset_index()
    )

    # Flag: distinguish real data from structural/sporadic zeros
    real_pairs = set(zip(raw["iso3"], raw["year"]))
    annual["ocha_data_flag"] = annual.apply(
        lambda r: "ocha_reported" if (r.iso3, r.year) in real_pairs
        else ("ocha_zero_structural" if r.iso3 in {"PRI", "VEN"}
              else "ocha_zero_no_flows"),
        axis=1
    )

    # ── QA ────────────────────────────────────────────────────────────────────
    print("\n=== QA SUMMARY ===")
    print(f"Rows: {len(annual)} (expected {len(ALL_COUNTRIES) * len(YEARS)} = {len(ALL_COUNTRIES)}×{len(YEARS)})")
    print(f"Nulls in funding_usd: {annual['funding_usd'].isna().sum()}")
    print(f"\nFlag breakdown:\n{annual['ocha_data_flag'].value_counts().to_string()}")

    print("\nFunding by country × year (USD millions):")
    pivot = (
        annual.pivot_table(index="iso3", columns="year", values="funding_usd")
        .fillna(0)
        .div(1e6)
        .round(1)
    )
    print(pivot.to_string())

    # ── Write country-year QA file ────────────────────────────────────────────
    annual.to_csv(OUT / "fts_country_year.csv", index=False)
    print(f"\n  Wrote fts_country_year.csv  ({len(annual)} rows)")

    # ── Expand to country-month ───────────────────────────────────────────────
    rows = []
    for _, r in annual.iterrows():
        monthly_funding = r.funding_usd / 12
        for month in pd.date_range(f"{int(r.year)}-01-01", f"{int(r.year)}-12-01", freq="MS"):
            rows.append({
                "iso3":            r.iso3,
                "year":            int(r.year),
                "month":           month,
                "funding_usd":     round(monthly_funding, 2),
                "ocha_data_flag":  r.ocha_data_flag,
            })

    monthly = pd.DataFrame(rows)
    monthly["month"] = pd.to_datetime(monthly["month"])
    monthly = monthly.sort_values(["iso3", "month"]).reset_index(drop=True)

    monthly.to_csv(OUT / "fts_country_month.csv", index=False)
    print(f"  Wrote fts_country_month.csv ({len(monthly)} rows)")

    # ── Coverage reminder ─────────────────────────────────────────────────────
    print("\n=== NOTES ===")
    print("  PRI: zero across all years — FTS does not track US territories")
    print("  VEN: zero across all years — origin country, not aid recipient in FTS")
    print("  Sporadic zeros: BLZ(2019), BRB(2019), JAM(2019/22/23), BHS(2022), SUR(2019)")
    print("  → All zeros filled; ocha_data_flag distinguishes type downstream")


if __name__ == "__main__":
    main()

Loading OCHA FTS files...
  2019: 13 countries found
  2020: 17 countries found
  2021: 17 countries found
  2022: 15 countries found
  2023: 16 countries found
  2024: 17 countries found

=== QA SUMMARY ===
Rows: 114 (expected 114 = 19×6)
Nulls in funding_usd: 0

Flag breakdown:
ocha_data_flag
ocha_reported           95
ocha_zero_structural    12
ocha_zero_no_flows       7

Funding by country × year (USD millions):
year   2019   2020   2021   2022   2023   2024
iso3                                          
BHS    30.2    4.6    0.4    0.0    0.4    0.5
BLZ     0.0    1.6    0.7    5.0    4.8    4.2
BRB     0.0    1.9    0.2    1.5    0.8    0.7
COL   309.3  520.4  507.0  488.5  439.6  463.0
CRI     4.3    9.8   16.9   22.9   25.8   23.5
CUB     3.3    6.0    1.0   10.7   10.3   13.0
DOM     0.5   14.2    6.5    9.0   13.4   13.3
GTM    22.7   54.8  118.6   88.6   69.6   85.7
GUY     3.4   10.2    2.3    5.0    3.1    2.7
HND     5.2   45.8  143.0  137.4  116.4  124.0
HTI    80.7  220